# Home Credit post-release frontier: categorical identity

**Controlled negative evidence, not a leaderboard claim.** This notebook replays recorded metrics from the 22 September 2026 run; it does not retrain models. The frozen September release remains unchanged.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"
pio.templates.default = "none"
R = Path("../reports/categorical_identity")
m = pd.read_csv(R / "fold_metrics.csv")
c = json.loads((R / "confirmation_decision.json").read_text())
u = json.loads((R / "conditional_uncertainty.json").read_text())
r = json.loads((R / "run_manifest.json").read_text())
summary = (
    f"run={r['run_id']} status={r['status']} "
    f"fits={r['fit_count_this_invocation']} "
    f"hours={r['elapsed_seconds'] / 3600:.2f}"
)
print(summary)

run=20260922T220711895557Z status=COMPLETE_CANDIDATE_REJECTED_ON_CONFIRMATION fits=13 hours=2.53


## Question and design

The released LightGBM path uses training-only category frequencies. The experiment tested native category identity and identity+frequency representations on the same 700-feature snapshot. Folds 1–3 selected a candidate; that candidate was frozen before folds 4–5. These are previously explored development periods, not a fresh external test.

In [2]:
models = [
    "saved_champion",
    "frequency",
    "blend_native",
    "native",
    "blend_dual",
    "dual",
]
focus = m[m.model.isin(models)].copy()
for name in models:
    p = focus[focus.model == name].sort_values("fold")
    if not p.empty:
        stability = [round(x, 6) for x in p.stability]
        auc = [round(x, 6) for x in p.auc]
        print(name, "stability", stability, "auc", auc)

saved_champion stability [0.472514, 0.670723, 0.491566, 0.68568, 0.689011] auc [0.848913, 0.842052, 0.852589, 0.850184, 0.864525]
frequency stability [0.46431, 0.671836, 0.457905, 0.669019, 0.688893] auc [0.848586, 0.842517, 0.851747, 0.848964, 0.864701]
blend_native stability [0.498806, 0.669847, 0.553191, 0.688476, 0.68634] auc [0.848526, 0.841701, 0.852324, 0.850333, 0.863011]
native stability [0.490739, 0.629313, 0.659533, 0.647508, 0.64847] auc [0.826227, 0.821767, 0.83403, 0.833995, 0.846218]
blend_dual stability [0.49309, 0.670168, 0.551982] auc [0.848684, 0.841833, 0.852303]
dual stability [0.49139, 0.630797, 0.659078] auc [0.82634, 0.822308, 0.834142]


## Fold stability

`blend_native` looked promising on the first three folds, which justified confirmation. The later windows are shown rather than hidden inside a mean.

In [3]:
fig = go.Figure()
for name in ["saved_champion", "frequency", "blend_native", "native"]:
    p = focus[focus.model == name].sort_values("fold")
    if not p.empty:
        fig.add_trace(
            go.Scatter(
                x=p.fold,
                y=p.stability,
                mode="lines+markers",
                name=name,
            )
        )
fig.update_layout(
    title="Official stability by temporal fold",
    xaxis_title="Fold",
    yaxis_title="Weekly-Gini stability",
    height=480,
    width=900,
)
fig.update_xaxes(dtick=1)
fig.show()

## Frozen confirmation gate

The selected `blend_native` candidate had only **+0.000062** mean stability versus the saved champion on folds 4–5, with one win and one loss. Its worst fold delta was **−0.002671**, so the predeclared champion gate failed. The descriptive moving-block interval also crossed zero.

In [4]:
rows = []
for ref, comparison in c["comparisons"].items():
    rows.append(
        (
            ref,
            comparison["mean_delta"],
            comparison["wins"],
            comparison["worst_delta"],
            comparison["passed"],
        )
    )
print("reference | mean_delta | wins | worst_delta | passed")
for row in rows:
    print(f"{row[0]} | {row[1]:+.6f} | {row[2]}/2 | {row[3]:+.6f} | {row[4]}")
interval = u["saved_champion"]["conditional_interval_95"]
print("saved-champion descriptive interval:", [round(x, 6) for x in interval])

reference | mean_delta | wins | worst_delta | passed
matched_frequency | +0.008452 | 1/2 | -0.002553 | True
saved_champion | +0.000062 | 1/2 | -0.002671 | False
saved-champion descriptive interval: [-0.0132, 0.036385]


## Decision and capability gained

**Reject the candidate. Do not refit or submit it.** Native categorical identity is now a tested negative direction rather than an untested gap. The next post-release research should target a genuinely missing capability—previous-application category occurrence histograms or a complementary DenseLight neural challenger—rather than another LightGBM encoding variation.

In [5]:
print(r["decision"])
print("FRONTIER_NOTEBOOK_COMPLETED")

Do not refit or submit this candidate; later development windows failed the predeclared gate.
FRONTIER_NOTEBOOK_COMPLETED
